# Dashboards from the cleaned data

- the question - Do equity funds with a higher expense ratio return less, and how large is the gap between direct and regular plans?


In [1]:
# imports
try:
    import dash,dash_bootstrap_components
except ImportError:
    !pip install dash dash-bootstrap-components
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dash import Dash, dcc, html, Input, Output
import dash_bootstrap_components as dbc

# Dashboard1

In [2]:
nav = pd.read_parquet('data_collection/clean/nav_midcap_clean.parquet')
ter = pd.read_parquet('data_collection/clean/ter_midcap_clean.parquet')
DIRECT,REGULAR = "#38BDF8","#FB923C"
COUNT = "#A78BFA"
GRID = "#343B46"
CARDS = "#282828"
TEXT = "#E6EDF3"
MUTED = "#9AA7B4"
THEME = dbc.themes.CYBORG
print(ter.info())
print("-"*40) 
print(nav.info())  
print("-"*40)
print(ter["MF_ID"].nunique())
print(nav["MF_ID"].nunique())

<class 'pandas.DataFrame'>
RangeIndex: 68236 entries, 0 to 68235
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Scheme_Name        68236 non-null  string        
 1   R_BaseTER          62315 non-null  float64       
 2   R_6A_B             62315 non-null  float64       
 3   R_6A_C             62315 non-null  float64       
 4   R_GST              62315 non-null  float64       
 5   R_TER              68172 non-null  float64       
 6   D_BaseTER          62315 non-null  float64       
 7   D_6A_B             62315 non-null  float64       
 8   D_6A_C             62315 non-null  float64       
 9   D_GST              62315 non-null  float64       
 10  D_TER              67341 non-null  float64       
 11  MF_ID              68236 non-null  int64         
 12  R_BER              5921 non-null   float64       
 13  R_BrokerageCost    5921 non-null   float64       
 14  R_TransactionCost

In [3]:
ter = ter[ter["MF_ID"].isin(nav["MF_ID"].unique())].copy()
companies = (nav.drop_duplicates(subset=["MF_ID"])[["MF_ID","Mutual_Fund_Company"]].assign(COMPANY=lambda x: x["Mutual_Fund_Company"].str.replace("Mutual Fund","",regex=False).str.strip()))
print(companies.info())
print(companies)

<class 'pandas.DataFrame'>
Index: 33 entries, 0 to 93746
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   MF_ID                33 non-null     int64
 1   Mutual_Fund_Company  33 non-null     str  
 2   COMPANY              33 non-null     str  
dtypes: int64(1), str(2)
memory usage: 2.0 KB
None
       MF_ID                Mutual_Fund_Company                COMPANY
0          3  Aditya Birla Sun Life Mutual Fund  Aditya Birla Sun Life
3822       4     Baroda BNP Paribas Mutual Fund     Baroda BNP Paribas
6066       6                    DSP Mutual Fund                    DSP
9888       9                   HDFC Mutual Fund                   HDFC
13710     13                  quant Mutual Fund                  quant
17534     16           JM Financial Mutual Fund           JM Financial
19432     17         Kotak Mahindra Mutual Fund         Kotak Mahindra
23254     18                    LIC Mutual Fund     

- KPI Cards

In [4]:
# ---------- numbers for the overview ----------
latest = (ter.dropna(subset=["R_TER", "D_TER"]).sort_values("Date").groupby("MF_ID").tail(1))
n_funds = nav["MF_ID"].nunique()
n_rows  = len(nav)
years   = (nav["Date"].max() - nav["Date"].min()).days / 365.25
avg_r, avg_d = latest["R_TER"].mean(), latest["D_TER"].mean()
avg_gap = (latest["R_TER"] - latest["D_TER"]).mean()

# one row per fund COMPANY: first NAV date, last NAV date
span = (nav.groupby("MF_ID")["Date"].agg(start="min", end="max").reset_index().merge(companies, on="MF_ID"))
span["years_of_history"] = (span["end"] - span["start"]).dt.days / 365.25
span["month"] = span["start"].dt.to_period("M").dt.to_timestamp()

# how many fund companies have a NAV in each month, and who joined that month
active = (nav.groupby(nav["Date"].dt.to_period("M"))["MF_ID"].nunique().rename("funds").reset_index())
active["Date"] = active["Date"].dt.to_timestamp()
joined = span.groupby("month")["COMPANY"].apply(lambda s: "<br>".join("• " + h for h in sorted(s)))
active["note"] = [("Joined:<br>" + joined[d]) if d in joined.index else "" for d in active["Date"]]

first_n, last_n = int(active["funds"].iloc[0]), int(active["funds"].iloc[-1])
active.loc[0, "note"] = f"{first_n} companies were already in the data"

newcomers = span[span["start"] > nav["Date"].min()]              # not the day-one group
busiest_year  = int(newcomers["start"].dt.year.value_counts().idxmax())
busiest_count = int(newcomers["start"].dt.year.value_counts().max())
newest = span.sort_values("start").iloc[-1]

COMPANY_options = [{"label": h, "value": str(i)}
                 for i, h in sorted(zip(companies["MF_ID"], companies["COMPANY"]), key=lambda x: x[1].lower())]
default_id = int(next((o["value"] for o in COMPANY_options if o["label"] == "HDFC"), COMPANY_options[0]["value"]))

In [5]:
# ---------- helpers ----------
def style_fig(fig, height=380):
    fig.update_layout(
        template="plotly_dark",
        paper_bgcolor=CARDS,
        plot_bgcolor=CARDS,

        font=dict(
            family="Inter, Segoe UI, sans-serif",
            color=TEXT,
            size=13
        ),

        height=height,

        margin=dict(l=50, r=35, t=25, b=45),

        hoverlabel=dict(
            bgcolor="#1B2430",
            bordercolor=GRID,
            font=dict(color=TEXT, size=14)
        ),

        legend=dict(
            orientation="h",
            y=1.12,
            x=0
        )
    )

    fig.update_xaxes(
        showgrid=False,
        zeroline=False,
        tickfont=dict(color=MUTED)
    )

    fig.update_yaxes(
        gridcolor=GRID,
        zeroline=False,
        tickfont=dict(color=MUTED)
    )

    return fig

def graph(fig, height=380, **kw):
    return dcc.Graph(figure=style_fig(fig, height), config={"displayModeBar": False}, **kw)

def kpi(label, value, color=None, sub=None):
    body = [html.P(label, className="text-muted mb-1"),
            html.H3(value, style={"color": color} if color else {})]
    if sub: body.append(html.Small(sub, className="text-muted"))
    return dbc.Card(dbc.CardBody(body), className="h-100")

def kpi_two(label, d_val, r_val, sub=None):
    """A card that shows the Direct and Regular value on two lines."""
    line = lambda name, val, col: html.Div([html.Span(name + "  ", className="text-muted"),
                                            html.Span(val, style={"color": col, "fontWeight": 600, "fontSize": "1.5rem"})])
    body = [html.P(label, className="text-muted mb-1"), line("Direct", d_val, DIRECT), line("Regular", r_val, REGULAR)]
    if sub: body.append(html.Small(sub, className="text-muted"))
    return dbc.Card(dbc.CardBody(body), className="h-100")

def chart_card(title, subtitle, content):
    return dbc.Card(dbc.CardBody([html.H5(title, className="mb-0"),
                                  html.Small(subtitle, className="text-muted"), content]), className="h-100")

fmt = lambda x, f: "n/a" if pd.isna(x) else format(x, f)

# ---------- chart 1: how many fund COMPANYs over time ----------
ends = active.iloc[[0, -1]]
fig_active = go.Figure()
fig_active.add_trace(go.Scatter(
    x=active["Date"], y=active["funds"], mode="lines",
    line=dict(color=COUNT, width=3, shape="hv"),
    fill="tozeroy", fillcolor="#3C384A",
    customdata=active["note"],
    hovertemplate="<b>%{y} fund companies</b> · %{x|%b %Y}<br>%{customdata}<extra></extra>"))
fig_active.add_trace(go.Scatter(                   # direct labels on the first and last point
    x=ends["Date"], y=ends["funds"], mode="markers+text", text=ends["funds"].astype(str),
    textposition="top center", textfont=dict(color=COUNT, size=18),
    marker=dict(color=COUNT, size=10), hoverinfo="skip", showlegend=False))
fig_active.update_yaxes(range=[0, last_n * 1.25], dtick=10)
fig_active.update_layout(showlegend=False)

# ---------- the TER chart: monthly average, lightly smoothed, labelled directly ----------
def make_ter_fig(t):
    fig = go.Figure()
    tm = (t.set_index("Date")[["D_TER", "R_TER"]].resample("MS").mean()
            .dropna(how="all").rolling(3, min_periods=1).mean())
    both = tm.dropna()
    if both.empty:                                   # a COMPANY with no TER data
        fig.add_annotation(text="No TER data for this COMPANY", showarrow=False,
                           font=dict(color="#8B98A5", size=14))
        return fig

    last = both.iloc[-1]
    last_x = both.index[-1]
    gap_now = last["R_TER"] - last["D_TER"]

    fig.add_trace(go.Scatter(x=tm.index, y=tm["D_TER"], name="Direct",
                             line=dict(color=DIRECT, width=3, shape="spline", smoothing=0.6),
                             hovertemplate="%{y:.2f}%"))
    fig.add_trace(go.Scatter(x=tm.index, y=tm["R_TER"], name="Regular",
                             line=dict(color=REGULAR, width=3, shape="spline", smoothing=0.6),
                             fill="tonexty", fillcolor="#4A392B",
                             hovertemplate="%{y:.2f}%"))
    # direct labels at the end of each line; cliponaxis=False lets them sit outside the plot area
    for name, col, val in [("Regular", REGULAR, last["R_TER"]), ("Direct", DIRECT, last["D_TER"])]:
        fig.add_trace(go.Scatter(x=[last_x], y=[val], mode="markers+text",
                                 text=[f"  {name} {val:.2f}%"], textposition="middle right",
                                 textfont=dict(color=col, size=13), marker=dict(color=col, size=9),
                                 cliponaxis=False, hoverinfo="skip", showlegend=False))
    fig.add_annotation(x=last_x, y=(last["R_TER"] + last["D_TER"]) / 2,
                       text=f"  gap {gap_now:.2f} pp", showarrow=False, xanchor="left",
                       font=dict(color="#8B98A5", size=12))

    fig.update_layout(hovermode="x unified", showlegend=False, margin=dict(r=150))
    fig.update_xaxes(range=[tm.index[0], last_x])    # stop the axis at the last data point (no empty 2028)
    fig.update_yaxes(ticksuffix="%", tickformat=".1f", rangemode="tozero")
    return fig

# ---------- the profile of one fund COMPANY ----------
def profile(mf_id):
    n = nav[nav["MF_ID"] == mf_id].dropna(subset=["NAV"]).sort_values("Date")
    t = ter[ter["MF_ID"] == mf_id].sort_values("Date")
    COMPANY = companies.loc[companies["MF_ID"] == mf_id, "COMPANY"].iloc[0]
    scheme = t["scheme_label"].iloc[-1] if ("scheme_label" in t and len(t)) else ""
    sp = span[span["MF_ID"] == mf_id].iloc[0]

    def nav_stats(plan):
        s = n.loc[n["Plan"] == plan, "NAV"]
        if s.empty: return dict(start=np.nan, now=np.nan, avg=np.nan, chg=np.nan)
        return dict(start=s.iloc[0], now=s.iloc[-1], avg=s.mean(), chg=(s.iloc[-1] / s.iloc[0] - 1) * 100)

    def ter_stats(col):
        s = t[col].dropna()
        if s.empty: return dict(start=np.nan, now=np.nan, avg=np.nan, lo=np.nan, hi=np.nan)
        return dict(start=s.iloc[0], now=s.iloc[-1], avg=s.mean(), lo=s.min(), hi=s.max())

    nd, nr = nav_stats("Direct Plan"), nav_stats("Regular Plan")
    td, tr_ = ter_stats("D_TER"), ter_stats("R_TER")

    title = html.Div([html.H3(COMPANY, className="mb-0"), html.Span(scheme, className="text-muted")])

    cards = dbc.Row([
        dbc.Col(kpi("In our data since", f"{sp['start']:%d %b %Y}", sub=f"{sp['years_of_history']:.1f} years of history"), width=3),
        dbc.Col(kpi_two("NAV now (₹ per unit)", fmt(nd["now"], ",.2f"), fmt(nr["now"], ",.2f"), f"as of {n['Date'].max():%d %b %Y}"), width=3),
        dbc.Col(kpi_two("TER now (% a year)", fmt(td["now"], ".2f") + "%", fmt(tr_["now"], ".2f") + "%",
                        f"gap {fmt(tr_['now'] - td['now'], '.2f')} pp"), width=3),
        dbc.Col(kpi_two("Average TER", fmt(td["avg"], ".2f") + "%", fmt(tr_["avg"], ".2f") + "%", "over its whole history"), width=3),
    ], className="mb-3")

    # NAV chart
    fig_nav = go.Figure()
    for plan, col in [("Direct Plan", DIRECT), ("Regular Plan", REGULAR)]:
        s = n[n["Plan"] == plan]
        fig_nav.add_trace(go.Scatter(x=s["Date"], y=s["NAV"], name=plan.replace(" Plan", ""),
                                     line=dict(color=col, width=2), hovertemplate="%{y:,.2f}"))
    fig_nav.update_layout(hovermode="x unified")
    fig_nav.update_yaxes(title_text="NAV (₹)")

    # TER chart (new)
    fig_ter = make_ter_fig(t)

    # numbers table (color="dark" removed so the theme styles it)
    rows = [("NAV at start (₹)",      fmt(nd["start"], ",.2f"), fmt(nr["start"], ",.2f")),
            ("NAV now (₹)",           fmt(nd["now"], ",.2f"),   fmt(nr["now"], ",.2f")),
            ("NAV change since start", fmt(nd["chg"], "+,.0f") + "%", fmt(nr["chg"], "+,.0f") + "%"),
            ("Average NAV (₹)",       fmt(nd["avg"], ",.2f"),   fmt(nr["avg"], ",.2f")),
            ("TER at start",          fmt(td["start"], ".2f") + "%", fmt(tr_["start"], ".2f") + "%"),
            ("TER now",               fmt(td["now"], ".2f") + "%",   fmt(tr_["now"], ".2f") + "%"),
            ("Average TER",           fmt(td["avg"], ".2f") + "%",   fmt(tr_["avg"], ".2f") + "%"),
            ("TER lowest to highest", f"{fmt(td['lo'], '.2f')}% to {fmt(td['hi'], '.2f')}%",
                                      f"{fmt(tr_['lo'], '.2f')}% to {fmt(tr_['hi'], '.2f')}%")]
    table = dbc.Table([
        html.Thead(html.Tr([html.Th(""), html.Th("Direct", style={"color": DIRECT}), html.Th("Regular", style={"color": REGULAR})])),
        html.Tbody([html.Tr([html.Td(a, className="text-muted"), html.Td(b), html.Td(c)]) for a, b, c in rows])],
        hover=True, className="mb-0")
    return title, cards, fig_nav, fig_ter, table

In [7]:
app = Dash(__name__, external_stylesheets=[THEME])

title0, cards0, fig_nav0, fig_ter0, table0 = profile(default_id)     # first view is drawn without a callback

app.layout = dbc.Container([
    # ---- header ----
    dbc.Row(dbc.Col([html.H1("Mid-cap mutual funds: Direct vs Regular", className="mt-3 mb-0"),
                     html.P("Dashboard 1 · Who is in our data, and what does each fund COMPANY look like?",
                            className="text-muted")], width=12), className="mb-3"),

    # ---- overview KPIs (2+2+2+2+2+2) ----
    dbc.Row([
        dbc.Col(kpi("Fund COMPANYs", f"{n_funds}"), width=2),
        dbc.Col(kpi("NAV rows", f"{n_rows:,}"), width=2),
        dbc.Col(kpi("Years of data", f"{years:.1f}"), width=2),
        dbc.Col(kpi("Avg Regular TER", f"{avg_r:.2f}%", REGULAR), width=2),
        dbc.Col(kpi("Avg Direct TER", f"{avg_d:.2f}%", DIRECT), width=2),
        dbc.Col(kpi("Avg gap", f"{avg_gap:.2f} pp"), width=2),
    ], className="mb-4"),

    # ---- how many funds over time (8 + 4) ----
    dbc.Row([
        dbc.Col(chart_card("How many mid-cap fund COMPANYs are in our data?",
                           f"Grew from {first_n} to {last_n}. Hover a step to see who joined.",
                           graph(fig_active, 340)), width=8),
        dbc.Col(dbc.Card(dbc.CardBody([
            html.H5("What this tells us"),
            dcc.Markdown(f"""
- **{first_n}** COMPANYs were there on day one. Now there are **{last_n}** (**+{last_n - first_n}**).
- **{busiest_year}** brought the most new COMPANYs (**{busiest_count}**).
- The newest is **{newest['COMPANY']}**, first seen {newest['start']:%d %b %Y}.
- Younger funds have less history, so later dashboards compare funds on **common time windows**.
            """)]), className="h-100"), width=4),
    ], className="mb-4"),

    # ---- fund COMPANY explorer ----
    dbc.Row([
        dbc.Col([html.H4("Explore a fund COMPANY", className="mb-0"),
                 html.Small("Pick a COMPANY to see when it started, its NAV and TER, now and on average.", className="text-muted")], md=8),
        dbc.Col(dcc.Dropdown(id="COMPANY-select", options=COMPANY_options, value=str(default_id), clearable=False), md=4),
    ], className="mb-3 align-items-end"),

    dbc.Row(dbc.Col(html.Div(title0, id="COMPANY-title"), width=12), className="mb-2"),
    html.Div(cards0, id="COMPANY-cards"),

    dbc.Row([
        dbc.Col(chart_card("NAV over time", "Price of one unit. Direct sits slightly above Regular.",
                           dcc.Graph(id="nav-graph", figure=style_fig(fig_nav0, 340), config={"displayModeBar": False})), width=6),
        dbc.Col(chart_card("TER over time", "Yearly fee. The shaded gap is what the distributor's commission costs.",
                           dcc.Graph(id="ter-graph", figure=style_fig(fig_ter0, 340), config={"displayModeBar": False})), width=6),
    ], className="mb-4"),

    dbc.Row(dbc.Col(chart_card("The numbers behind the charts", "Direct vs Regular for this COMPANY", html.Div(table0, id="COMPANY-table")), width=12),
            className="mb-5"),
], fluid=True, className="px-4")

# one input -> five outputs (his "Pattern 3")
@app.callback(Output("COMPANY-title", "children"), Output("COMPANY-cards", "children"),
              Output("nav-graph", "figure"), Output("ter-graph", "figure"), Output("COMPANY-table", "children"),
              Input("COMPANY-select", "value"))
def update_COMPANY(value):
    title, cards, f_nav, f_ter, table = profile(int(value))
    return title, cards, style_fig(f_nav, 340), style_fig(f_ter, 340), table

app.run(jupyter_mode="inline", jupyter_height=2300, port=8060, debug=True, dev_tools_hot_reload=False)

# Dashboard 2

In [13]:
# ---------- Dashboard 2: what does the commission cost? ----------
GAP = "#F472B6"          # pink = the gap (Regular TER minus Direct TER)

# one row per fund company per date, with the gap
t2 = ter.dropna(subset=["R_TER", "D_TER"]).drop(columns=["COMPANY"], errors="ignore").copy()
t2["gap"] = t2["R_TER"] - t2["D_TER"]
t2 = t2.merge(companies[["MF_ID", "COMPANY"]], on="MF_ID").sort_values("Date")
t2["year"] = t2["Date"].dt.year
t2["month"] = t2["Date"].dt.to_period("M").dt.to_timestamp()

# ---------- KPI numbers: latest reading of each company ----------
now2 = t2.groupby("MF_ID").tail(1)
first2 = t2.groupby("MF_ID").head(1)
avg_gap2 = now2["gap"].mean()
share = avg_gap2 / now2["R_TER"].mean() * 100
widest = now2.loc[now2["gap"].idxmax()]
narrowest = now2.loc[now2["gap"].idxmin()]
n_narrowed = int((now2.set_index("MF_ID")["gap"].sort_index() < first2.set_index("MF_ID")["gap"].sort_index()).sum())
n_total = len(now2)

# ---------- chart A: median gap over time ----------
by_month = t2.groupby(["MF_ID", "month"])["gap"].mean().reset_index()
monthly = by_month.groupby("month")["gap"].agg(["median", "count"]).reset_index()
monthly = monthly[monthly["count"] >= 5]          # need at least 5 companies in the month

fig_gap = px.line(monthly, x="month", y="median", color_discrete_sequence=[GAP])
fig_gap.update_traces(line_width=3)
fig_gap.update_xaxes(title_text="")
fig_gap.update_yaxes(title_text="Median gap (pp)", rangemode="tozero")

# ---------- chart B: gap of every company in one year ----------
year_gap = t2.groupby(["COMPANY", "year"])["gap"].mean().reset_index()
y0, y1 = int(year_gap["year"].min()), int(year_gap["year"].max())
gmax = year_gap["gap"].max()

def rank_fig(year):
    d = year_gap[year_gap["year"] == year].sort_values("gap")     # biggest ends up on top
    fig = px.bar(d, x="gap", y="COMPANY", orientation="h", text_auto=".2f",
                 color_discrete_sequence=[GAP])
    fig.update_traces(textposition="outside", cliponaxis=False)
    fig.update_xaxes(range=[0, gmax * 1.15], title_text="Gap (pp)")   # same range every year
    fig.update_yaxes(title_text="")
    return style_fig(fig, max(420, 26 * len(d) + 60))

# ---------- chart C: what an investment grows to ----------
def calc(amount, n_years, gross):
    rows = []
    for y in range(n_years + 1):
        rows.append({"Year": y, "Plan": "Direct", "Value": amount * (1 + (gross - avg_d) / 100) ** y})
        rows.append({"Year": y, "Plan": "Regular", "Value": amount * (1 + (gross - avg_r) / 100) ** y})
    d = pd.DataFrame(rows)
    fig = px.line(d, x="Year", y="Value", color="Plan",
                  color_discrete_map={"Direct": DIRECT, "Regular": REGULAR})
    fig.update_traces(line_width=3)
    fig.update_yaxes(title_text="Value (Rs lakh)", rangemode="tozero")
    direct = d[(d["Plan"] == "Direct") & (d["Year"] == n_years)]["Value"].iloc[0]
    regular = d[(d["Plan"] == "Regular") & (d["Year"] == n_years)]["Value"].iloc[0]
    text = (f"On **Rs {amount} lakh** at **{gross}%** a year before costs, after **{n_years} years** "
            f"Direct is worth **Rs {direct:.1f} lakh** and Regular **Rs {regular:.1f} lakh**. "
            f"The commission costs about **Rs {direct - regular:.1f} lakh** "
            f"({(direct - regular) / direct * 100:.0f}% of the Direct value).")
    return style_fig(fig, 340), text


In [14]:
app2 = Dash(__name__, external_stylesheets=[THEME])

fig_calc0, text_calc0 = calc(10, 15, 12)

# small helper-free slider marks in readable grey
def grey(values):
    return {v: {"label": str(v), "style": {"color": MUTED}} for v in values}

app2.layout = dbc.Container([
    dbc.Row(dbc.Col([html.H1("Mid-cap mutual funds: Direct vs Regular", className="mt-3 mb-0"),
                     html.P("Dashboard 2 · What does the distributor's commission cost?", className="text-muted")],
                    width=12), className="mb-3"),

    dbc.Row([
        dbc.Col(kpi("Average gap today", f"{avg_gap2:.2f} pp", GAP, "Regular TER minus Direct TER"), width=3),
        dbc.Col(kpi("Commission share", f"{share:.0f}%", GAP, "of the Regular TER is the gap"), width=3),
        dbc.Col(kpi("Widest gap", f"{widest['gap']:.2f} pp", sub=widest["COMPANY"]), width=3),
        dbc.Col(kpi("Narrowest gap", f"{narrowest['gap']:.2f} pp", sub=narrowest["COMPANY"]), width=3),
    ], className="mb-4"),

    dbc.Row([
        dbc.Col(chart_card("How the gap has moved over time",
                           "The middle fund company's gap, month by month.",
                           dcc.Graph(figure=style_fig(fig_gap, 340), config={"displayModeBar": False})), width=8),
        dbc.Col(dbc.Card(dbc.CardBody([
            html.H5("What this tells us"),
            dcc.Markdown(f"""
- On average, **{avg_gap2:.2f} pp** of the Regular fee goes to the distributor, about **{share:.0f}%** of it.
- The gap **narrowed in {n_narrowed} of {n_total}** fund companies since their first reading.
- **{widest['COMPANY']}** has the widest gap ({widest['gap']:.2f} pp), **{narrowest['COMPANY']}** the narrowest ({narrowest['gap']:.2f} pp).
- A small yearly gap still adds up, because it compounds. See the calculator below.
            """)]), className="h-100"), width=4),
    ], className="mb-4"),

    dbc.Row([
        # ---- left: ranking with a year slider ----
        dbc.Col(dbc.Card(dbc.CardBody([
            html.H5("Gap by fund company", className="mb-0"),
            html.Small("Average Regular minus Direct TER in the chosen year. Drag the slider.", className="text-muted"),
            dcc.Slider(id="rank-year", min=y0, max=y1, step=1, value=y1, marks=grey(range(y0, y1 + 1))),
            dcc.Graph(id="rank-graph", figure=rank_fig(y1), config={"displayModeBar": False}),
        ])), width=6),

        # ---- right: cost calculator ----
        dbc.Col(dbc.Card(dbc.CardBody([
            html.H5("What the gap costs over the years", className="mb-0"),
            html.Small("Uses today's average Direct and Regular TER of all fund companies.", className="text-muted"),
            html.Label("Amount invested (Rs lakh)", className="text-muted mt-3"),
            dcc.Slider(id="amount", min=1, max=100, step=1, value=10, marks=grey([1, 25, 50, 75, 100])),
            html.Label("Years invested", className="text-muted mt-3"),
            dcc.Slider(id="years", min=1, max=30, step=1, value=15, marks=grey([1, 5, 10, 15, 20, 25, 30])),
            html.Label("Return before costs (% a year)", className="text-muted mt-3"),
            dcc.Slider(id="gross", min=6, max=18, step=1, value=12, marks=grey([6, 9, 12, 15, 18])),
            dcc.Graph(id="calc-graph", figure=fig_calc0, config={"displayModeBar": False}),
            dcc.Markdown(text_calc0, id="calc-text"),
        ])), width=6),
    ], className="mb-5"),
], fluid=True, className="px-4")

@app2.callback(Output("rank-graph", "figure"), Input("rank-year", "value"))
def update_rank(year):
    return rank_fig(year)

@app2.callback(Output("calc-graph", "figure"), Output("calc-text", "children"),
               Input("amount", "value"), Input("years", "value"), Input("gross", "value"))
def update_calc(amount, n_years, gross):
    return calc(amount, n_years, gross)

app2.run(jupyter_mode="inline", jupyter_height=2000, port=8062, debug=True, dev_tools_hot_reload=False)


# Dashboard 3

In [15]:
# ---------- Dashboard 3: do Direct plans really earn more? ----------
# one row per company and date, with both plans' NAV side by side
pv = (nav.pivot_table(index=["MF_ID", "Date"], columns="Plan", values="NAV")
        .rename(columns={"Direct Plan": "Direct", "Regular Plan": "Regular"})
        .dropna().reset_index().sort_values(["MF_ID", "Date"]))
pv = pv.merge(companies[["MF_ID", "COMPANY"]], on="MF_ID")
last_date = pv["Date"].max()

WINDOWS = {"1 year": 1, "3 years": 3, "5 years": 5, "Since start": None}

def window_stats(n_years):
    """Yearly return of each plan over the last n_years (or the whole history if None)."""
    rows = []
    for mf, g in pv.groupby("MF_ID"):
        end = g["Date"].iloc[-1]
        if end < last_date - pd.Timedelta(days=30):          # stopped reporting
            continue
        if n_years is not None:
            start_date = end - pd.DateOffset(years=n_years)
            g = g[g["Date"] >= start_date]
            if (g["Date"].iloc[0] - start_date).days > 10:   # not enough history
                continue
        span_years = (end - g["Date"].iloc[0]).days / 365.25
        if span_years < 0.9:
            continue
        d = ((g["Direct"].iloc[-1] / g["Direct"].iloc[0]) ** (1 / span_years) - 1) * 100
        r = ((g["Regular"].iloc[-1] / g["Regular"].iloc[0]) ** (1 / span_years) - 1) * 100
        fee = t2[(t2["MF_ID"] == mf) & (t2["Date"] >= g["Date"].iloc[0])]["gap"].mean()
        rows.append({"COMPANY": g["COMPANY"].iloc[0], "Direct": d, "Regular": r,
                     "ret_gap": d - r, "fee_gap": fee})
    return pd.DataFrame(rows)

STATS = {name: window_stats(n) for name, n in WINDOWS.items()}

# ---------- KPI row ----------
def kpi_row(st):
    ahead = int((st["ret_gap"] > 0).sum())
    return dbc.Row([
        dbc.Col(kpi("Median Direct return", f"{st['Direct'].median():.1f}%", DIRECT, "a year"), width=3),
        dbc.Col(kpi("Median Regular return", f"{st['Regular'].median():.1f}%", REGULAR, "a year"), width=3),
        dbc.Col(kpi("Median return gap", f"{st['ret_gap'].median():.2f} pp", GAP, "Direct minus Regular, per year"), width=3),
        dbc.Col(kpi("Direct ahead in", f"{ahead} of {len(st)}", sub="fund companies in this window"), width=3),
    ], className="mb-4")

# ---------- chart A: return of each company, Direct vs Regular ----------
def return_fig(st):
    d = st.sort_values("Direct")                                   # best Direct return ends up on top
    long = d.melt(id_vars="COMPANY", value_vars=["Regular", "Direct"], var_name="Plan", value_name="Return")
    fig = px.bar(long, x="Return", y="COMPANY", color="Plan", barmode="group", orientation="h",
                 color_discrete_map={"Direct": DIRECT, "Regular": REGULAR})
    fig.update_yaxes(categoryorder="array", categoryarray=list(d["COMPANY"]), title_text="")
    fig.update_xaxes(ticksuffix="%", title_text="Return (% a year)")
    return style_fig(fig, 40 * len(d) + 100)

# ---------- chart B: does the fee gap show up in the returns? ----------
def scatter_fig(st):
    s = st.dropna(subset=["fee_gap"])
    lo = min(0, s["fee_gap"].min(), s["ret_gap"].min())
    hi = max(s["fee_gap"].max(), s["ret_gap"].max())
    fig = px.scatter(s, x="fee_gap", y="ret_gap", hover_name="COMPANY", color_discrete_sequence=[GAP])
    fig.update_traces(marker_size=11)
    fig.add_trace(go.Scatter(x=[lo, hi], y=[lo, hi], mode="lines", name="return gap = fee gap",
                             line=dict(color=MUTED, dash="dot")))
    fig.update_xaxes(title_text="Fee gap (pp a year)", showgrid=True, gridcolor=GRID)
    fig.update_yaxes(title_text="Return gap (pp a year)")
    fig.update_layout(showlegend=False)
    return style_fig(fig, 380)

def window_text(st, name):
    return f"""
- **{len(st)}** fund companies have enough history for **{name.lower()}**. The others are left out.
- Direct earned a median **{st['ret_gap'].median():.2f} pp** a year more than Regular.
- The fee gap (TER) over the same period had a median of **{st['fee_gap'].median():.2f} pp**.
- A dot on the dotted line means the fee gap explains the return gap fully.
"""

# ---------- chart C: what Rs 10 lakh really became ----------
def growth_fig(mf):
    g = pv[pv["MF_ID"] == mf].copy()
    g["Direct"] = g["Direct"] / g["Direct"].iloc[0] * 10           # Rs lakh
    g["Regular"] = g["Regular"] / g["Regular"].iloc[0] * 10
    long = g.melt(id_vars="Date", value_vars=["Direct", "Regular"], var_name="Plan", value_name="Value")
    fig = px.line(long, x="Date", y="Value", color="Plan",
                  color_discrete_map={"Direct": DIRECT, "Regular": REGULAR})
    fig.update_traces(line_width=3)
    fig.update_xaxes(title_text="")
    fig.update_yaxes(title_text="Value (Rs lakh)")
    fig.update_layout(hovermode="x unified")
    behind = g["Direct"].iloc[-1] - g["Regular"].iloc[-1]
    text = (f"Rs 10 lakh put into **{g['COMPANY'].iloc[0]}** on **{g['Date'].iloc[0]:%d %b %Y}** is worth "
            f"**Rs {g['Direct'].iloc[-1]:.1f} lakh** in Direct and **Rs {g['Regular'].iloc[-1]:.1f} lakh** in Regular "
            f"on **{g['Date'].iloc[-1]:%d %b %Y}**. Regular is behind by **Rs {behind:.1f} lakh**.")
    return style_fig(fig, 380), text


In [16]:
app3 = Dash(__name__, external_stylesheets=[THEME])

name_of = companies.set_index("MF_ID")["COMPANY"].to_dict()
co_ids = sorted(pv["MF_ID"].unique(), key=lambda m: name_of[m].lower())
co_options = [{"label": name_of[m], "value": int(m)} for m in co_ids]
co_default = int(next((m for m in co_ids if name_of[m] == "HDFC"), co_ids[0]))

W0 = "3 years"
fig_g0, text_g0 = growth_fig(co_default)

app3.layout = dbc.Container([
    dbc.Row(dbc.Col([html.H1("Mid-cap mutual funds: Direct vs Regular", className="mt-3 mb-0"),
                     html.P("Dashboard 3 · Do Direct plans really earn more?", className="text-muted")],
                    width=12), className="mb-3"),

    # ---- window picker ----
    dbc.Row([
        dbc.Col([html.H4("Pick a time window", className="mb-0"),
                 html.Small("Only companies with enough history for the window are compared.", className="text-muted")], md=6),
        dbc.Col(dcc.RadioItems(id="window", options=list(WINDOWS), value=W0, inline=True,
                               labelStyle={"marginRight": "20px"}), md=6),
    ], className="mb-3 align-items-end"),

    html.Div(kpi_row(STATS[W0]), id="kpis"),

    dbc.Row([
        dbc.Col(chart_card("Return of each fund company",
                           "Blue = Direct, orange = Regular. The gap between the two bars is what Regular gives up.",
                           dcc.Graph(id="return-graph", figure=return_fig(STATS[W0]), config={"displayModeBar": False})),
                width=7),
        dbc.Col([
            chart_card("Does the fee gap show up in returns?", "Each dot is a fund company.",
                       dcc.Graph(id="scatter-graph", figure=scatter_fig(STATS[W0]), config={"displayModeBar": False})),
            dbc.Card(dbc.CardBody([html.H5("What this tells us"),
                                   dcc.Markdown(window_text(STATS[W0], W0), id="ret-text")]), className="mt-3"),
        ], width=5),
    ], className="mb-4"),

    # ---- one company, real growth ----
    dbc.Row([
        dbc.Col([html.H4("What Rs 10 lakh really became", className="mb-0"),
                 html.Small("Pick a fund company to see its full history, Direct vs Regular.", className="text-muted")], md=8),
        dbc.Col(dcc.Dropdown(id="co-select", options=co_options, value=co_default, clearable=False), md=4),
    ], className="mb-3 align-items-end"),
    dbc.Row(dbc.Col(chart_card("Growth of Rs 10 lakh", "Same money, same day, two plans.",
                               html.Div([dcc.Graph(id="growth-graph", figure=fig_g0, config={"displayModeBar": False}),
                                         dcc.Markdown(text_g0, id="growth-text")])), width=12),
            className="mb-5"),
], fluid=True, className="px-4")

@app3.callback(Output("kpis", "children"), Output("return-graph", "figure"),
               Output("scatter-graph", "figure"), Output("ret-text", "children"),
               Input("window", "value"))
def update_window(name):
    st = STATS[name]
    return kpi_row(st), return_fig(st), scatter_fig(st), window_text(st, name)

@app3.callback(Output("growth-graph", "figure"), Output("growth-text", "children"), Input("co-select", "value"))
def update_growth(mf):
    return growth_fig(int(mf))

app3.run(jupyter_mode="inline", jupyter_height=2300, port=8063, debug=True, dev_tools_hot_reload=False)
